In [ ]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import sklearn
import tensorflow as tf
import time
from pathlib import Path
from collections import Counter
import cv2
from sklearn.model_selection import train_test_split

path = Path('../Data/classification')

In [ ]:
import shutil

output_base = Path('../Data/split')
splits = ['train', 'val', 'test']

for cls in path.iterdir():
    if not cls.is_dir():
        continue

    images = [f for f in cls.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]

    train_files, temp = train_test_split(images, test_size=0.30, random_state=42)
    val_files, test_files = train_test_split(temp, test_size=0.50, random_state=42)

    for split, files in zip(splits, [train_files, val_files, test_files]):
        dest = output_base / split / cls.name
        dest.mkdir(parents=True, exist_ok=True)
        for f in files:
            shutil.copy(f, dest / f.name)

for split in splits:
    counts = {cls.name: len(list(cls.iterdir())) for cls in (output_base / split).iterdir()}
    print(f"{split}: {counts}")

train: {'glioma': 1134, 'meningioma': 1151, 'notumor': 1400, 'pituitary': 1229}
val: {'glioma': 243, 'meningioma': 247, 'notumor': 300, 'pituitary': 264}
test: {'glioma': 244, 'meningioma': 247, 'notumor': 300, 'pituitary': 264}


In [ ]:
# TASK 1.3 — Augment training data
# Way 1 (used here): manually save augmented images with cv2 — blur, noise, flip
# Way 2 (alternative): tf.keras.Sequential([RandomFlip(), RandomZoom(), ...]) inside the model
#   → augmentation happens on-the-fly during training, no extra files saved
#   → simpler but you can't inspect the augmented images

train_base = output_base / 'train'

def add_noise(img):
    noise = np.random.normal(0, 15, img.shape).astype(np.uint8)
    return cv2.add(img, noise)

for cls_folder in train_base.iterdir():
    if not cls_folder.is_dir():
        continue
    for img_path in list(cls_folder.iterdir()):
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        cv2.imwrite(str(cls_folder / (img_path.stem + '_blur' + img_path.suffix)),
                    cv2.GaussianBlur(img, (5, 5), 0))

        cv2.imwrite(str(cls_folder / (img_path.stem + '_noise' + img_path.suffix)),
                    add_noise(img))

        cv2.imwrite(str(cls_folder / (img_path.stem + '_flip' + img_path.suffix)),
                    cv2.flip(img, 1))

counts_after = {cls.name: len(list(cls.iterdir())) for cls in train_base.iterdir()}
print('train after augmentation:', counts_after)

train after augmentation: {'glioma': 4536, 'meningioma': 4604, 'notumor': 5600, 'pituitary': 4916}


In [ ]:
# TASK 1.4 — Build the TensorFlow data pipeline

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    output_base / 'train',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    output_base / 'val',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    output_base / 'test',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

preprocess = tf.keras.applications.inception_v3.preprocess_input

train_ds = train_ds.map(lambda x, y: (preprocess(x), y)).cache().prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (preprocess(x), y)).cache().prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.map(lambda x, y: (preprocess(x), y)).cache().prefetch(tf.data.AUTOTUNE)

class_names = train_ds.class_names
print('Classes:', class_names)
print('Train batches:', len(train_ds))
print('Val batches:  ', len(val_ds))
print('Test batches: ', len(test_ds))

In [ ]:
# TASK 2.1 — Build the model (Feature Extraction)

NUM_CLASSES = len(class_names)

base_model = tf.keras.applications.InceptionV3(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Alternative loss if labels are one-hot encoded instead of integers:
# loss='categorical_crossentropy'

model.summary()

In [ ]:
# TASK 2.2 — Train with MLflow tracking (Feature Extraction phase)

import mlflow
import mlflow.keras

mlflow.set_tracking_uri('http://mlflow:5000')
# if running locally outside Docker:

mlflow.set_experiment('brain-tumor-classification')

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    '../models/feature_extraction_best.keras', monitor='val_loss', save_best_only=True
)

with mlflow.start_run(run_name='feature_extraction'):

    mlflow.log_params({
        'learning_rate': 1e-3,
        'batch_size': BATCH_SIZE,
        'epochs': 30,
        'base_model': 'InceptionV3',
        'frozen': True
    })

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        callbacks=[early_stop, checkpoint]
    )

    for epoch, (acc, val_acc, loss, val_loss) in enumerate(zip(
        history.history['accuracy'],
        history.history['val_accuracy'],
        history.history['loss'],
        history.history['val_loss']
    )):
        mlflow.log_metrics({'accuracy': acc, 'val_accuracy': val_acc,
                            'loss': loss, 'val_loss': val_loss}, step=epoch)

    mlflow.keras.log_model(model, 'model')

In [ ]:
# TASK 2.3 => Evaluate the model

from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model = tf.keras.models.load_model('../models/feature_extraction_best.keras')

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))

ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names)
plt.tight_layout()
plt.savefig('../models/confusion_matrix_feature_extraction.png')
plt.show()

with mlflow.start_run(run_name='feature_extraction', nested=True):
    mlflow.log_metrics({
        'test_accuracy' : accuracy_score(y_true, y_pred),
        'test_precision': precision_score(y_true, y_pred, average='weighted'),
        'test_recall'   : recall_score(y_true, y_pred, average='weighted'),
        'test_f1'       : f1_score(y_true, y_pred, average='weighted')
    })
    mlflow.log_artifact('../models/confusion_matrix_feature_extraction.png')